<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m4_01_linear_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M4.01 — Straight lines through the traffic

In the lab we fitted a linear regression and a logistic regression and
read what they learned: one weight per feature and an intercept. In this
workbook you do the same **on the traffic of a motorway**. You will read
a model's weights in real units, watch a single wrong number in the data
wipe out an effect, find out why the largest weight is not the most
important feature, and meet a classifier that never says "yes".

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "model_busiest_hour": "535fa30d7e25dd8a",
    "predicted_class": "91c6fc73ce4953a7",
    "probabilities_sum": "6b86b273ff34fce1",
    "rain_effect": "aad50a2ea3f6c9c6",
    "robust_model_fixed_it": "9390298f3fb0c5b1",
    "snow_rank": "5e94f29b0eaf2c86",
    "temperature_alone": "1d287684996990dd",
    "times_the_coefficient": "4a44dc15364204a8",
    "value_at_zero": "2ba9ed81ae9fd812"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

A sensor counts the vehicles driving west on Interstate 94 between
Minneapolis and Saint Paul (USA). Each row is **one hour** of 2016 or
2017:

- `traffic_volume`: number of vehicles during that hour (our target);
- `temp`: temperature in **kelvin**; `rain_1h` and `snow_1h`: millimetres
  of rain and snow during the hour; `clouds_all`: cloud cover in percent;
- `weather_main`: a one-word description of the weather;
- `holiday`: the name of the public holiday, or `No holiday`;
- `date_time`: the hour the row describes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

traffic = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/metro_traffic.csv",
    parse_dates=["date_time"],
)
traffic["temp_c"] = traffic["temp"] - 273.15
traffic["hour"] = traffic["date_time"].dt.hour
print(f"{len(traffic)} hours, mean traffic {traffic['traffic_volume'].mean():.0f} vehicles per hour")
traffic.head()

We added two columns: `temp_c`, the temperature in degrees Celsius, and
`hour`, the hour of the day (0 to 23).

## 2. A linear regression with one feature

Do people drive more when it is warm? A linear regression with a single
feature draws the straight line

`predicted traffic = intercept + coefficient × temperature`.

In [ ]:
from sklearn.linear_model import LinearRegression

temperature_model = LinearRegression()
temperature_model.fit(traffic[["temp_c"]], traffic["traffic_volume"])
coefficient, intercept = temperature_model.coef_[0], temperature_model.intercept_
print(f"predicted traffic = {intercept:.0f} + {coefficient:.1f} × temp_c")

sample = traffic.sample(n=2000, random_state=0)
line = pd.DataFrame({"temp_c": [-30, 35]})
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.scatter(sample["temp_c"], sample["traffic_volume"], s=4, alpha=0.4)
ax.plot(line["temp_c"], temperature_model.predict(line), color="black", linewidth=2)
_ = ax.set(xlabel="temperature (°C)", ylabel="vehicles per hour")

The **coefficient** has a unit: vehicles per hour **per degree**. It is
how much the prediction changes when the temperature rises by one degree.

✏️ At a temperature of exactly **0 °C**, the predicted traffic is equal
to one of the two numbers of the model. Which one? Write `"intercept"`
or `"coefficient"`.

In [ ]:
value_at_zero = ____
check("value_at_zero", value_at_zero)

✏️ Two hours differ by **10 °C**. By how many times the coefficient do
their predicted traffic volumes differ?

In [ ]:
times_the_coefficient = ____
check("times_the_coefficient", times_the_coefficient)

Is this a useful model? As always, we compare with a baseline that
predicts the mean traffic for every hour. (Both errors below are measured
on the data used for training. With two numbers to learn from 16 551
hours, that makes no difference here.)

In [ ]:
from sklearn.metrics import mean_absolute_error

baseline_error = mean_absolute_error(
    traffic["traffic_volume"], np.full(len(traffic), traffic["traffic_volume"].mean())
)
temperature_error = mean_absolute_error(
    traffic["traffic_volume"], temperature_model.predict(traffic[["temp_c"]])
)
print(f"always predicting the mean: error {baseline_error:.0f} vehicles per hour")
print(f"line on the temperature:    error {temperature_error:.0f} vehicles per hour")

✏️ How useful is the temperature **alone** for predicting traffic? Write
`"almost useless"` or `"very useful"`.

In [ ]:
temperature_alone = ____
check("temperature_alone", temperature_alone)

🧠 **Think.** The coefficient is clearly positive, the line clearly goes
up, and yet the model barely beats the baseline. Look at the cloud of
points: what does a coefficient tell you, and what does it not tell you?

## 3. ➕ Going further: one wrong number

Does rain keep drivers at home? Same model, with `rain_1h` (millimetres
of rain during the hour) as the only feature.

🔮 **Predict first.** Will the coefficient of the rain be **"clearly
negative"** (less traffic when it rains), **"about zero"** or **"clearly
positive"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["clearly negative", "about zero", "clearly positive"])

In [ ]:
rain_model = LinearRegression().fit(traffic[["rain_1h"]], traffic["traffic_volume"])
print(f"coefficient of the rain: {rain_model.coef_[0]:.2f} vehicles per hour per mm")

Practically zero: according to this model, rain changes nothing. Before
believing it, look at the feature:

In [ ]:
traffic["rain_1h"].describe().round(2)

Three quarters of the hours have no rain at all, and the maximum is
**9831 mm in one hour**: almost ten metres of water. The rainiest places
on Earth receive about that much in a whole year. It is a sensor or
typing error, in a single row:

In [ ]:
traffic.sort_values("rain_1h", ascending=False).head(3)[["date_time", "rain_1h", "weather_main", "traffic_volume"]]

We fit the same model again **without that one row**, and draw both
lines.

In [ ]:
plausible = traffic[traffic["rain_1h"] < 100]
clean_rain_model = LinearRegression().fit(plausible[["rain_1h"]], plausible["traffic_volume"])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
panels = [
    (axes[0], traffic, rain_model, "all rows"),
    (axes[1], plausible, clean_rain_model, "without the impossible reading"),
]
for ax, rows, fitted, title in panels:
    axis = pd.DataFrame({"rain_1h": [0, rows["rain_1h"].max()]})
    ax.scatter(rows["rain_1h"], rows["traffic_volume"], s=5, alpha=0.4)
    ax.plot(axis["rain_1h"], fitted.predict(axis), color="black", linewidth=2)
    ax.set(title=f"{title}: coefficient {fitted.coef_[0]:.1f}", xlabel="rain during the hour (mm)")
_ = axes[0].set_ylabel("vehicles per hour")

✏️ Once the impossible reading is removed, what does the model say about
rain? Write `"less traffic"`, `"no effect"` or `"more traffic"`.

In [ ]:
rain_effect = ____
check("rain_effect", rain_effect)

**One row out of 16 551 decided the result.** On the left plot you can
see why: a line has to pass near a point that sits alone, thousands of
millimetres away from all the others, so that point alone fixes the
slope.

### Does a "robust" model save us?

scikit-learn has linear models designed to resist outliers.
`HuberRegressor` is one: it gives less weight to rows whose **target**
is far from the prediction.

🔮 **Predict first.** Trained on **all** rows, impossible reading
included, will `HuberRegressor` find the negative effect of rain?
`"yes"` or `"no"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["yes", "no"])

In [ ]:
from sklearn.linear_model import HuberRegressor

robust_model = HuberRegressor(max_iter=1000).fit(traffic[["rain_1h"]], traffic["traffic_volume"])
print(f"coefficient of the rain, HuberRegressor on all rows: {robust_model.coef_[0]:.2f}")

✏️ Did the robust model fix the problem? Write `"yes"` or `"no"`.

In [ ]:
robust_model_fixed_it = ____
check("robust_model_fixed_it", robust_model_fixed_it)

The traffic during that hour was perfectly ordinary: the strange value is
in a **feature**, not in the target, and that is not what this model
protects against. No algorithm replaces looking at your data:
`describe()`, a plot, and the question "can this number exist?".

From here on we work without the impossible row.

🧠 **Think.** We removed the row. What else could we have done with it?
In which situation would removing rows be a bad idea?

## 4. ➕ Going further: which feature matters most?

A linear regression with four weather features learns four coefficients.
A tempting way to rank the features is by the size of their coefficient.

In [ ]:
weather_features = ["temp_c", "rain_1h", "snow_1h", "clouds_all"]
weather_model = LinearRegression().fit(plausible[weather_features], plausible["traffic_volume"])
coefficients = pd.DataFrame(
    {
        "coefficient": weather_model.coef_,
        "typical variation of the feature": plausible[weather_features].std().to_numpy(),
    },
    index=weather_features,
)
coefficients.round(3)

`snow_1h` has by far the largest coefficient. But a coefficient is "per
unit of the feature", and the features do not vary by the same amounts:
the second column gives the standard deviation of each one. Snow was
recorded during only a handful of hours, and never more than a fraction
of a millimetre.

The effect a feature really has on the predictions is its coefficient
**multiplied by how much the feature varies**.

🔮 **Predict first.** After this multiplication, where will `snow_1h`
rank among the four features? **"most important"** or **"least
important"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["most important", "least important"])

In [ ]:
coefficients["coefficient × variation"] = (
    coefficients["coefficient"] * coefficients["typical variation of the feature"]
)
coefficients.round(2)

✏️ Where does `snow_1h` rank in the last column (ignore the signs)? Write
`"most important"` or `"least important"`.

In [ ]:
snow_rank = ____
check("snow_rank", snow_rank)

This is the second reason, after module 1, to put a `StandardScaler`
before a linear model: once every feature varies by the same amount, the
coefficients can be compared with one another. scikit-learn's example
[Common pitfalls in the interpretation of coefficients of linear models](https://scikit-learn.org/1.8/auto_examples/inspection/plot_linear_model_coefficient_interpretation.html)
does this on the wages data of module 1.

## 5. Logistic regression: will the road be busy?

For the road operator, the useful question is a yes/no one: will the
traffic be **heavy** (more than 4500 vehicles in the hour)? That is a
classification problem, and logistic regression is the linear model for
it. We give it the hour of the day and the temperature.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

features = plausible[["hour", "temp_c"]]
heavy = plausible["traffic_volume"] > 4500
print(f"share of hours with heavy traffic: {heavy.mean():.2f}")

🔮 **Predict first.** A model that always answers "not heavy" is right
about 65% of the time. Everybody knows that traffic depends on the hour. Will
the logistic regression be **"no better than always answering not
heavy"**, **"a little better"** or **"much better"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["no better than always answering not heavy", "a little better", "much better"])

In [ ]:
classifier = make_pipeline(StandardScaler(), LogisticRegression())
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_validate(classifier, features, heavy, cv=cv)["test_score"]
print(f"accuracy of the logistic regression: {scores.mean():.3f} ± {scores.std():.3f}")
print(f"always answering 'not heavy':        {1 - heavy.mean():.3f}")

To understand, we look at what the model computes. A logistic regression
does not answer yes or no directly: it computes a **probability**, and
answers "yes" when that probability is above 0.5.

In [ ]:
classifier.fit(features, heavy)
examples = pd.DataFrame({"hour": [3, 8, 16, 23], "temp_c": [10.0, 10.0, 10.0, 10.0]})
probabilities = classifier.predict_proba(examples)
pd.DataFrame(
    {
        "hour": examples["hour"],
        "P(not heavy)": probabilities[:, 0],
        "P(heavy)": probabilities[:, 1],
        "prediction": classifier.predict(examples),
    }
).round(2)

✏️ In each row, what do the two probabilities add up to? And if a model
gives `P(heavy) = 0.44` for some hour, which class does it predict?
Write `"heavy"` or `"not heavy"`.

In [ ]:
probabilities_sum = ____
predicted_class = ____
check("probabilities_sum", probabilities_sum)
check("predicted_class", predicted_class, hint="Compare the probability of 'heavy' with the threshold.")

The probability of heavy traffic never reaches 0.5, so the model never
predicts it. Here is the probability it gives for each hour of the day
(at 10 °C), next to what really happens:

In [ ]:
day = pd.DataFrame({"hour": range(24), "temp_c": 10.0})
model_probability = classifier.predict_proba(day)[:, 1]
real_share = heavy.groupby(plausible["hour"]).mean()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(real_share.index, real_share, color="lightgray", label="real share of heavy hours")
ax.plot(day["hour"], model_probability, "o-", color="black", label="probability given by the model")
ax.axhline(0.5, linestyle="--", color="red", label="decision threshold")
ax.set(xlabel="hour of the day", ylabel="share / probability", xticks=range(0, 24, 2))
_ = ax.legend(loc="upper left", fontsize=8)

✏️ According to **the model**, at which hour of the day is heavy traffic
the most likely?

In [ ]:
model_busiest_hour = ____
check("model_busiest_hour", model_busiest_hour, hint="Read the black curve, not the grey bars.")

The model's answer is an hour at which heavy traffic **never** happens. A logistic regression multiplies `hour` by one weight: its
probability can only go **up all the way or down all the way** as the
hour grows. Real traffic goes up in the morning and down in the evening,
and no single weight can say that.

The model is not broken and it does not need more data. It needs the
hour to be **described differently**, and that is the subject of the
next workbook.

🧠 **Think.**

1. For a linear model, `hour = 23` is "23 times more" than `hour = 1`.
   In workbook M1.03 we met another encoding that invented an order that
   means nothing. Which one?
2. The accuracy of this model equals the baseline's **exactly**. Why is
   that not a coincidence?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - what the coefficient and the intercept of the temperature model
     mean, in units, and why the model is still almost useless;
   - the impossible rain reading: what it did to the model, and why the
     robust model did not help;
   - why the feature with the largest coefficient was the least
     important;
   - why the logistic regression never predicts heavy traffic.